In [ ]:
from __future__ import annotations
from typing import Any

from dataclasses import dataclass
from pathlib import Path
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
YLORBR_CMAP = plt.get_cmap("YlOrBr")
YLORBR_PALETTE = {
    "pale": YLORBR_CMAP(0.18),
    "light": YLORBR_CMAP(0.32),
    "mid": YLORBR_CMAP(0.50),
    "strong": YLORBR_CMAP(0.68),
    "dark": YLORBR_CMAP(0.86),
    "deep": YLORBR_CMAP(0.98),
}

In [ ]:
@dataclass
class ModelParameters1D:
    """Container for the 1-D vertical-column model parameters.

    :param time_steps: Number of explicit timesteps to simulate.
    :param dt_days: Duration of each timestep in days.
    :param initial_mat_thickness_mm: Starting thickness for each newly exposed mat.
    :param lamina_completion_thickness_mm: Legacy thickness threshold retained for comparison.
    :param surface_light: Mean relative incident light above the water column.
    :param light_amplitude: Annual sinusoidal amplitude applied to mean incident light.
    :param light_peak_day: Day of year when incident light reaches its maximum.
    :param min_surface_light: Minimum allowed incident light after seasonal forcing.
    :param light_attenuation_per_mm: Legacy column attenuation coefficient retained for comparison.
    :param water_depth_mm: Water level above the initial stromatolite base.
    :param minimum_water_cover_mm: Minimum water cover retained above the active mat surface.
    :param water_light_attenuation_per_mm: Beer-Lambert attenuation coefficient through water.
    :param mean_temperature_c: Mean annual temperature in degrees Celsius.
    :param temperature_amplitude_c: Annual sinusoidal temperature amplitude in degrees Celsius.
    :param temperature_peak_day: Day of year when temperature reaches its maximum.
    :param optimal_temperature_c: Temperature where biological rates are maximal.
    :param temperature_width_c: Width of the bell-shaped biological temperature response.
    :param days_per_year: Annual period used by seasonal forcing functions.
    :param max_growth_rate_per_day: Maximum photosynthetic growth rate.
    :param half_saturation_light: Light level giving half-maximal growth.
    :param mortality_rate_per_day: Biomass mortality rate in the active surface mat.
    :param buried_mortality_rate_per_day: Additional biomass mortality in buried layers.
    :param eps_production_rate_per_day: EPS production rate per unit biomass.
    :param eps_decay_rate_per_day: EPS decay or compaction rate.
    :param sediment_input_mm_per_day: Baseline potential sediment supplied from the water column.
    :param sediment_seasonal_amplitude: Annual sinusoidal amplitude applied to baseline sediment supply.
    :param sediment_peak_day: Day of year when seasonal sediment supply reaches its maximum.
    :param min_sediment_multiplier: Minimum allowed seasonal sediment multiplier.
    :param storm_probability_per_day: Probability of a rare storm sediment pulse on any simulated day.
    :param storm_sediment_mm_per_day: Mean additional sediment supply during a storm pulse.
    :param storm_noise_fraction: Relative random variation in storm pulse size.
    :param trapping_efficiency: Maximum fraction of supplied sediment retained by EPS.
    :param eps_half_saturation: EPS level giving half-maximal sediment trapping.
    :param biomass_to_thickness: Conversion from biomass growth to thickness.
    :param eps_to_thickness: Conversion from EPS gain to thickness.
    :param sediment_to_thickness: Conversion from trapped sediment to thickness.
    :param burial_sediment_threshold_mm: Cumulative trapped sediment cover needed to bury a mat.
    :param rapid_burial_sediment_threshold_mm: Single-timestep trapped sediment pulse that triggers burial.
    :param new_layer_biomass_seed: Biomass used to seed a new surface mat.
    :param new_layer_eps_seed: EPS used to seed a new surface mat.
    :param random_seed: Seed for reproducible sediment-supply variation.
    :param sediment_noise_fraction: Fractional random variation in sediment supply.
    """

    time_steps: int
    dt_days: float
    initial_mat_thickness_mm: float
    lamina_completion_thickness_mm: float
    surface_light: float
    light_amplitude: float
    light_peak_day: float
    min_surface_light: float
    light_attenuation_per_mm: float
    water_depth_mm: float
    minimum_water_cover_mm: float
    water_light_attenuation_per_mm: float
    mean_temperature_c: float
    temperature_amplitude_c: float
    temperature_peak_day: float
    optimal_temperature_c: float
    temperature_width_c: float
    days_per_year: float
    max_growth_rate_per_day: float
    half_saturation_light: float
    mortality_rate_per_day: float
    buried_mortality_rate_per_day: float
    eps_production_rate_per_day: float
    eps_decay_rate_per_day: float
    sediment_input_mm_per_day: float
    sediment_seasonal_amplitude: float
    sediment_peak_day: float
    min_sediment_multiplier: float
    storm_probability_per_day: float
    storm_sediment_mm_per_day: float
    storm_noise_fraction: float
    trapping_efficiency: float
    eps_half_saturation: float
    biomass_to_thickness: float
    eps_to_thickness: float
    sediment_to_thickness: float
    burial_sediment_threshold_mm: float
    rapid_burial_sediment_threshold_mm: float
    new_layer_biomass_seed: float
    new_layer_eps_seed: float
    random_seed: int
    sediment_noise_fraction: float

In [ ]:
def load_parameter_values(parameter_path: Path) -> dict:
    """Load named parameter values from a JSON config file.

    :param parameter_path: Path to a JSON config file with a parameters object.
    :return: Mapping from parameter names to raw values.
    """
    with parameter_path.open() as handle:
        parameter_document = json.load(handle)

    return {
        name: parameter["value"]
        for name, parameter in parameter_document["parameters"].items()
    }

In [ ]:
def day_of_year(step: int, dt_days: float, days_per_year: float) -> float:
    """Convert a timestep into a repeating day-of-year value.

    :param step: Current simulation step.
    :param dt_days: Duration of each timestep in days.
    :param days_per_year: Number of days in one seasonal cycle.
    :return: Day within the seasonal cycle.
    """
    return (step * dt_days) % days_per_year

In [ ]:
def seasonal_light(day: float, params: Any) -> float:
    """Compute seasonally forced incident light for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean light, amplitude, peak day, and minimum light.
    :return: Relative incident light after seasonal forcing and lower clamping.
    """
    phase = 2.0 * math.pi * (day - params.light_peak_day) / params.days_per_year
    forced_light = params.surface_light * (1.0 + params.light_amplitude * math.cos(phase))
    return max(params.min_surface_light, forced_light)

In [ ]:
def seasonal_temperature(day: float, params: Any) -> float:
    """Compute seasonally forced temperature for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean temperature, amplitude, and peak day.
    :return: Temperature in degrees Celsius.
    """
    phase = 2.0 * math.pi * (day - params.temperature_peak_day) / params.days_per_year
    return params.mean_temperature_c + params.temperature_amplitude_c * math.cos(phase)

In [ ]:
def temperature_response(temperature_c: float, optimal_temperature_c: float, temperature_width_c: float) -> float:
    """Compute biological suitability from temperature.

    :param temperature_c: Current temperature in degrees Celsius.
    :param optimal_temperature_c: Temperature where the multiplier reaches one.
    :param temperature_width_c: Width of the bell-shaped response curve.
    :return: Biological rate multiplier between zero and one.
    """
    scaled_difference = (temperature_c - optimal_temperature_c) / temperature_width_c
    return math.exp(-(scaled_difference ** 2))

In [ ]:
def light_limitation(light: np.ndarray, half_saturation_light: float) -> np.ndarray:
    """Compute the Monod-style light limitation factor.

    :param light: Local relative light intensity.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Photosynthetic light limitation factor between zero and one.
    """
    return light / (half_saturation_light + light)

In [ ]:
def monod_growth_rate(light: np.ndarray, max_growth_rate: float, half_saturation_light: float) -> np.ndarray:
    """Compute light-limited photosynthetic growth rates.

    :param light: Local relative light intensity at each surface cell.
    :param max_growth_rate: Maximum possible growth rate per day.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Growth rates per day for all surface cells.
    """
    return max_growth_rate * light_limitation(light, half_saturation_light)

In [ ]:
def trapping_fraction(eps: np.ndarray, max_efficiency: float, eps_half_saturation: float) -> np.ndarray:
    """Compute the fraction of supplied sediment trapped by EPS.

    :param eps: Relative EPS concentration in each active mat.
    :param max_efficiency: Maximum possible trapping fraction.
    :param eps_half_saturation: EPS level where trapping reaches half its maximum.
    :return: Fraction of incoming sediment retained by each cell.
    """
    return max_efficiency * eps / (eps_half_saturation + eps)

In [ ]:
def water_depth_above_surface(height_mm: np.ndarray, params: Any) -> np.ndarray:
    """Compute local water depth above each active mat surface.

    :param height_mm: Current stromatolite height across the x-y grid.
    :param params: Model parameters controlling water level and minimum water cover.
    :return: Water depth above each active mat in millimetres.
    """
    return np.maximum(params.minimum_water_cover_mm, params.water_depth_mm - height_mm)

In [ ]:
def light_at_surface(day: float, height_mm: np.ndarray, params: Any) -> dict:
    """Compute light reaching each active mat through local overlying water.

    :param day: Day within the seasonal cycle.
    :param height_mm: Current stromatolite height across the x-y grid.
    :param params: Model parameters controlling incident light and water attenuation.
    :return: Dictionary containing incident light, local water depth, and local mat light.
    """
    incident_light = seasonal_light(day, params)
    water_depth_mm = water_depth_above_surface(height_mm, params)
    mat_light = incident_light * np.exp(-params.water_light_attenuation_per_mm * water_depth_mm)
    return {
        "incident_light": incident_light,
        "water_depth_above_mat_mm": water_depth_mm,
        "light_at_mat": mat_light,
    }

In [ ]:
def seasonal_sediment_multiplier(day: float, params: Any) -> float:
    """Compute the slow seasonal multiplier for sediment supply.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling sediment seasonality and lower clamping.
    :return: Multiplicative sediment-supply factor.
    """
    phase = 2.0 * math.pi * (day - params.sediment_peak_day) / params.days_per_year
    multiplier = 1.0 + params.sediment_seasonal_amplitude * math.cos(phase)
    return max(params.min_sediment_multiplier, multiplier)

In [ ]:
def daily_sediment_weather_multiplier(noise_fraction: float, rng: np.random.Generator) -> float:
    """Draw a non-negative domain-wide weather multiplier for sediment supply.

    :param noise_fraction: Standard deviation of daily weather variation as a fraction of the mean.
    :param rng: NumPy random generator used for reproducible draws.
    :return: Daily multiplicative sediment-supply factor.
    """
    return max(0.0, rng.normal(loc=1.0, scale=noise_fraction))

In [ ]:
def storm_sediment_pulse(params: Any, rng: np.random.Generator) -> tuple[float, bool]:
    """Draw a rare storm sediment pulse for one timestep.

    :param params: Model parameters controlling storm probability and pulse size.
    :param rng: NumPy random generator used for reproducible storm timing and size.
    :return: Additional sediment supply in millimetres per day and whether a storm occurred.
    """
    storm_probability = min(1.0, params.storm_probability_per_day * params.dt_days)
    storm_today = rng.random() < storm_probability
    if not storm_today:
        return 0.0, False

    pulse = rng.normal(
        loc=params.storm_sediment_mm_per_day,
        scale=params.storm_sediment_mm_per_day * params.storm_noise_fraction,
    )
    return max(0.0, pulse), True

In [ ]:
def spatial_sediment_supply(day: float, params: Any, rng: np.random.Generator) -> dict:
    """Combine shared forcing with spatially variable sediment supply.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling all sediment-supply forcing terms.
    :param rng: NumPy random generator used for reproducible stochastic forcing.
    :return: Dictionary containing sediment supply components in millimetres per day.
    """
    seasonal_multiplier = seasonal_sediment_multiplier(day, params)
    weather_multiplier = daily_sediment_weather_multiplier(params.sediment_noise_fraction, rng)
    storm_pulse, storm_today = storm_sediment_pulse(params, rng)
    background_supply = params.sediment_input_mm_per_day * seasonal_multiplier * weather_multiplier
    total_supply = background_supply + storm_pulse
    spatial_multiplier = np.maximum(
        0.0,
        rng.normal(loc=1.0, scale=params.spatial_sediment_noise, size=(params.ny, params.nx)),
    )

    return {
        "sediment_baseline_mm_per_day": params.sediment_input_mm_per_day,
        "sediment_seasonal_multiplier": seasonal_multiplier,
        "sediment_weather_multiplier": weather_multiplier,
        "storm_sediment_mm_per_day": storm_pulse,
        "storm_today": storm_today,
        "sediment_supply_mm_per_day": total_supply,
        "local_sediment_supply_mm_per_day": total_supply * spatial_multiplier,
    }

In [ ]:
def history_as_array(history: list[dict], field: str) -> np.ndarray:
    """Extract a named scalar history field as a NumPy array.

    :param history: List of timestep summary dictionaries.
    :param field: Dictionary key to extract from each summary.
    :return: Numeric array containing the requested field over time.
    """
    return np.array([row.get(field, np.nan) for row in history], dtype=float)

In [ ]:
def surface_laplacian(values: np.ndarray, active_mask: np.ndarray) -> np.ndarray:
    """Compute a two-dimensional nearest-neighbour Laplacian within the circular mask.

    :param values: Surface values across the x-y grid.
    :param active_mask: Boolean raster where True cells are active.
    :return: Difference between local values and active-neighbour values.
    """
    filled = np.where(active_mask, values, 0.0)
    padded_values = np.pad(filled, pad_width=1, mode="edge")
    padded_mask = np.pad(active_mask, pad_width=1, mode="constant", constant_values=False)
    centre = padded_values[1:-1, 1:-1]

    laplacian = np.zeros_like(values, dtype=float)
    for neighbour_values, neighbour_mask in [
        (padded_values[:-2, 1:-1], padded_mask[:-2, 1:-1]),
        (padded_values[2:, 1:-1], padded_mask[2:, 1:-1]),
        (padded_values[1:-1, :-2], padded_mask[1:-1, :-2]),
        (padded_values[1:-1, 2:], padded_mask[1:-1, 2:]),
    ]:
        laplacian += np.where(neighbour_mask, neighbour_values - centre, 0.0)

    return np.where(active_mask, laplacian, 0.0)

In [ ]:
def apply_surface_smoothing(height_mm: np.ndarray, active_mask: np.ndarray, params: Any) -> np.ndarray:
    """Apply weak surface smoothing inside the circular footprint.

    :param height_mm: Current stromatolite height across the x-y grid.
    :param active_mask: Boolean raster where True cells are active.
    :param params: Model parameters controlling smoothing strength.
    :return: Smoothed height field with inactive cells preserved as NaN.
    """
    smoothing = params.lateral_smoothing_rate * surface_laplacian(height_mm, active_mask)
    smoothed = np.maximum(0.0, np.where(active_mask, height_mm, 0.0) + smoothing)
    return np.where(active_mask, smoothed, np.nan)

In [ ]:
def active_values(values: np.ndarray, active_mask: np.ndarray) -> np.ndarray:
    """Extract only active circular-footprint values from a model raster.

    :param values: Full x-y raster containing active and inactive cells.
    :param active_mask: Boolean raster where True cells are active.
    :return: One-dimensional array of active values.
    """
    return values[active_mask]

In [ ]:
def masked_for_plot(values: np.ndarray) -> np.ma.MaskedArray:
    """Mask NaN cells so inactive regions remain visually empty.

    :param values: Raster whose inactive cells are represented by NaN values.
    :return: Masked array suitable for Matplotlib plotting.
    """
    return np.ma.masked_invalid(values)

In [ ]:
def circular_plot_cmap() -> plt.Colormap:
    """Return a YlOrBr colormap with transparent inactive cells.

    :return: Copy of the project colormap with NaN values drawn transparently.
    """
    cmap = YLORBR_CMAP.copy()
    cmap.set_bad((1.0, 1.0, 1.0, 0.0))
    return cmap

In [ ]:
def selected_slice_index(params: Any) -> int:
    """Choose the y-index used for cross-sectional comparisons.

    :param params: Model parameters containing an optional requested slice index.
    :return: Valid y-index through the x-y grid.
    """
    if params.slice_y_index is None:
        return params.ny // 2
    return int(np.clip(params.slice_y_index, 0, params.ny - 1))

In [ ]:
def save_stratigraphy_rasters(stratigraphy: dict, output_path: Path) -> None:
    """Save the core 3-D stratigraphic raster arrays as a compressed NumPy archive.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param output_path: Destination NPZ path.
    :return: None. The NPZ file is written to disk.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_path, **stratigraphy)